In [ ]:
import jax
jax.config.update('jax_platform_name', 'cpu')
from gould_2026.save_to_cache import save_to_cache

import numpy as np
import json
import seaborn as sns
import matplotlib.pyplot as plt
import pandas
from gould_2026.stim_designer import OptimizationMethod
from gould_2026.utils import angle_between
from gould_2026.sim_stim import StimDirectionType
import scipy.stats
import io
from gould_2026.plotting import Palette, LINEWIDTH, EM, make_violinplot_inner_kws, paper_plot_context
from optimization_comparison import make_table_over_target_type


rng = np.random.default_rng(0)

violinplot_inner_kws = make_violinplot_inner_kws()


In [ ]:
closed=False
optimization_method=OptimizationMethod.LBFGS_UNCONSTRAINED
n_runs=1

output1_violinplots = None
output2_with_controls = None


In [ ]:
optimization_method = OptimizationMethod(optimization_method)

In [ ]:
stim_direction_types = (StimDirectionType.RANDOM_FEASIBLE, StimDirectionType.FIRST, StimDirectionType.ONES, StimDirectionType.RANDOM, StimDirectionType.NEG_ONES)
# stim_direction_types = ('ones', 'random')

l_df = make_table_over_target_type(n_runs=n_runs, stim_direction_types=stim_direction_types)


stim_direction_type_subs = {'first': 'Q_0', 'random_feasible': 'feasible', '-ones': 'negative', 'ones': 'dense', 'random': 'random'}
l_df['display_stim_direction_type'] = l_df['stim_direction_type'].replace(stim_direction_type_subs)


l_df['angles(s_obs,v)'] = l_df.l.apply(lambda l: angle_between(l['observed_s_hat'], l['v']))



In [ ]:


order = ('Q_0', 'negative', 'dense', 'random', 'feasible')
l_df.sort_values(by='display_stim_direction_type', inplace=True, key=lambda x: x.apply(order.index))

_d = {
    OptimizationMethod.LBFGS: 'normal',
    OptimizationMethod.LBFGS_UNCONSTRAINED: 'normal',
    OptimizationMethod.LBFGS_SPARSE_CONSTRAINED: 'normal',
    OptimizationMethod.LBFGS_POSITIVE_CONSTRAINED: 'normal',
    OptimizationMethod.RANDOM_MANY_NEURONS: 'many',
    OptimizationMethod.RANDOM_DENSE_GAUSSIAN: 'gaussian'
}
_d = _d | {k.value:v for k, v in _d.items()}

l_df['optim_method_display'] = l_df['optim_method'].map(_d)


sub_df = l_df[
    (l_df['closed'] == closed)
    &
    l_df['optim_method'].apply(lambda x: x in [optimization_method, OptimizationMethod.RANDOM_MANY_NEURONS])
]




In [ ]:
with paper_plot_context():

    fig, axs= plt.subplots(ncols=1, nrows=1, figsize=np.array([1,1])*2, squeeze=False, layout='constrained')


    ax: plt.Axes = axs[0, 0]
    metric_name = 'angles(s_obs,v)'
    palette = {'normal': '#00000000', 'many': 'gray'}
    sns.violinplot(sub_df, x='display_stim_direction_type', y=metric_name, hue='optim_method_display', orient='v', ax=ax, width=1, cut=0, density_norm='width',inner_kws = violinplot_inner_kws, palette=palette, order=order, legend=False)
    ax.set_xlabel('Target type')
    ax.set_ylabel('Error angle ($^\\circ$)')
    ax.spines[['right', 'top']].set_visible(False)

    ax.tick_params(axis='x', bottom=False, top=False, left=False, right=False)
    ax.set_yticks([0, 45, 90, 135, 180])

    for i, collection in enumerate(ax.collections):
        if hasattr(collection, 'get_facecolor'):
            if (collection.get_facecolor() == np.array([0,0,0,1])).all(): # in palette, 'normal' gets black
                collection.set_facecolor(Palette[order[i//len(palette)]])

    if output2_with_controls is not None:
        fig.savefig(output2_with_controls)



In [ ]:
with paper_plot_context():

    fig, axs= plt.subplots(ncols=1, nrows=1, figsize=np.array([1,1])*2, squeeze=False, layout='constrained')

    ax: plt.Axes = axs[0, 0]
    metric_name = 'angles(s_obs,v)'

    palette = {o: Palette[o] for o in order}

    sns.violinplot(sub_df[sub_df['optim_method_display'] != 'many'], x='display_stim_direction_type', y=metric_name, hue='display_stim_direction_type', orient='v', ax=ax, width=1, cut=0, density_norm='width',inner_kws = violinplot_inner_kws, order=order, legend=False, palette=palette, alpha=1, saturation=1)
    ax.set_xlabel('Target type')
    ax.set_ylabel('Error angle ($^\\circ$)')
    ax.spines[['right', 'top']].set_visible(False)

    ax.tick_params(axis='x', bottom=False, top=False, left=False, right=False)
    ax.set_yticks([0, 45, 90, 135, 180])


    # for i, collection in enumerate(ax.collections):
    #     if hasattr(collection, 'get_facecolor'):
    #         collection.set_facecolor(Palette[order[i]])



    if output1_violinplots is not None:
        fig.savefig(output1_violinplots)


In [ ]:

# test output file
test_result_file = io.StringIO()
stim_direction_types = sub_df['display_stim_direction_type'].unique()
for stim_direction_type in stim_direction_types:
    test_result_file.write(f"comparison for '{stim_direction_type}', optimized ('normal') vs random ('many') [.05, .5, .95]\n")

    to_compare = dict()
    for optim_method in sub_df['optim_method_display'].unique():
        x = sub_df[(sub_df['display_stim_direction_type'] == stim_direction_type) & (sub_df['optim_method_display'] == optim_method)][metric_name]
        a, b, c = np.quantile(x, [.05, .5, .95])
        test_result_file.write(f'{optim_method}: [{a:06.3f}, {b:06.3f}, {c:06.3f}]')
        threshold = .9
        centered_to_median = np.abs(x - np.quantile(x, .5))
        test_result_file.write(f' median to .9 = {float(np.quantile(centered_to_median, threshold)):06.3f}\n')
        to_compare[optim_method] = x

    test_result = scipy.stats.wilcoxon(to_compare['normal'], to_compare['many'])
    test_result_file.write(f'p = {test_result.pvalue}\n\n')


In [ ]:

x = sub_df[(sub_df['display_stim_direction_type'] == 'feasible') & (sub_df['optim_method_display'] == 'normal')][metric_name]
y = sub_df[(sub_df['display_stim_direction_type'] == 'dense') & (sub_df['optim_method_display'] == 'normal')][metric_name]
test_result = scipy.stats.wilcoxon(x, y)
test_result_file.write(f'feasible vs dense:\n')
test_result_file.write(f'p = {test_result.pvalue}\n\n')

x = sub_df[(sub_df['display_stim_direction_type'] == 'feasible') & (sub_df['optim_method_display'] == 'normal')][metric_name]
y = sub_df[(sub_df['display_stim_direction_type'] == 'negative') & (sub_df['optim_method_display'] == 'normal')][metric_name]
test_result = scipy.stats.wilcoxon(x, y)
test_result_file.write(f'feasible vs negative:\n')
test_result_file.write(f'p = {test_result.pvalue}\n\n')


In [ ]:


fig_temp, ax = plt.subplots(ncols=1, nrows=1, figsize=np.array([1,1])*2, squeeze=True, layout='constrained', subplot_kw={'projection': '3d'})
sub_df['s_norm'] = sub_df['l'].apply(lambda l: np.linalg.norm(l['s']))
sub_df['angles(s,v)'] = sub_df.l.apply(lambda l: angle_between(l['s'], l['v']))

ax.plot(sub_df['angles(s,v)'], sub_df['angles(s_obs,v)'], sub_df['s_norm'], '.')
ax.set_xlabel('angles(s,v)')
ax.set_ylabel('angles(s_obs,v)')
ax.set_zlabel('s_norm')
# sns.scatterplot(data=sub_df, x='angles(s,v)', y='angles(s_obs,v)', hue='s_norm', ax=ax, legend=False)
plt.show()


In [ ]:
# save
fig, [], [test_result_file]
